In [1]:
import os
import ast
import numpy as np
import pandas as pd
import pydicom
from glob import glob
import nibabel as nib
import dicom2nifti

CSV_LOCALIZERS_PATH = "train_localizers.csv"
DICOM_DIR = "/Users/nicolas/Desktop/tochange"
REF_NII_PATH = "/Users/nicolas/Desktop/ref_image.nii.gz"
OUT_PATH = "/Users/nicolas/Desktop/mask1.nii.gz"


# 1) read CSV
df = pd.read_csv(CSV_LOCALIZERS_PATH)

# 2) load dicoms for one series and sort
dicom_files = glob(f"{DICOM_DIR}/*.dcm")
ds_list = [pydicom.dcmread(f, stop_before_pixels=True) for f in dicom_files] # Reads the metadata of all dicom files

In [2]:
# robust sort (prefer ImagePositionPatient, fallback InstanceNumber)
def sort_key(ds):
    if hasattr(ds, "ImagePositionPatient"):
        return float(ds.ImagePositionPatient[2]) # Z-coordinate of the silice's physical poisition in patient coordinates
    return int(ds.InstanceNumber) # Index identifying the position of the an image within a series 

ds_list.sort(key=sort_key)

In [3]:
sop_to_z = {ds.SOPInstanceUID: i for i, ds in enumerate(ds_list)}

In [4]:
# Get the In-plane spacing (row, col) => dy, dx
dy, dx = map(float, ds_list[0].PixelSpacing) # Get the physical space between pixel centers in mm/pixel

# Slice spacing: Prefer slice deltas if available (ImagePositionPatient)
if hasattr(ds_list[0], "SpacingBetweenSlices"):
    dz = float(ds_list[0].SpacingBetweenSlices) # Actual distance of the center (along slice axis) of adjacent slices
else:
    dz = float(getattr(ds_list[0], "SliceThickness", 1.0))

In [5]:
# This asumes that the volume is a series of dicom files (no multiframe dicom for now)

Z = len(ds_list # Number of dicom files (number of slices)
Y = int(ds_list[0].Rows) # Image height in pixels
X = int(ds_list[0].Columns) # Image width in pixels 
mask = np.zeros((Z, Y, X), dtype=np.uint8) # Makes a mask of the same size of the image

In [6]:
# Paint sphere:

def paint_sphere_mm(mask3d, cz, cy, cx, r_mm, dz, dy, dx, value=1):
    # convert physical radius to voxel extents
    rz = int(np.ceil(r_mm / dz))
    ry = int(np.ceil(r_mm / dy))
    rx = int(np.ceil(r_mm / dx))

    z0 = max(0, cz - rz)
    z1 = min(mask3d.shape[0] - 1, cz + rz)
    y0 = max(0, cy - ry)
    y1 = min(mask3d.shape[1] - 1, cy + ry)
    x0 = max(0, cx - rx)
    x1 = min(mask3d.shape[2] - 1, cx + rx)

    zz, yy, xx = np.ogrid[z0:z1+1, y0:y1+1, x0:x1+1]

    # squared distance in mm
    dist2 = ((zz - cz) * dz)**2 + ((yy - cy) * dy)**2 + ((xx - cx) * dx)**2
    sph = dist2 <= (r_mm * r_mm)

    mask3d[z0:z1+1, y0:y1+1, x0:x1+1][sph] = value    


In [7]:
R_mm = 1.5  # radius in millimeters
for _, row in df.iterrows():
    sop = row["SOPInstanceUID"]
    if sop not in sop_to_z:
        continue

    xy = ast.literal_eval(row["coordinates"])
    x = int(round(xy["x"]))
    y = int(round(xy["y"]))
    z = sop_to_z[sop]

    if 0 <= x < X and 0 <= y < Y and 0 <= z < Z:
        paint_sphere_mm(mask, z, y, x, R_mm, dz, dy, dx, value=1)

In [8]:
# Convert the DICOM series to a reference NIfTI (to get affine/header)

dicom_dir = DICOM_DIR
ref_nii_path = REF_NII_PATH
dicom2nifti.dicom_series_to_nifti(dicom_dir, ref_nii_path, reorient_nifti=True)

ref_img = nib.load(ref_nii_path)
ref_affine = ref_img.affine
ref_header = ref_img.header.copy()

mask_xyz = np.transpose(mask, (2, 1, 0))  # Assumes that reorient_nifti=True gives (Z,Y,X) and needs to be transformed to (X,Y,Z)

# Make sure shapes match, otherwise fix ordering/reorient choice
print("ref shape:", ref_img.shape, "mask shape:", mask_xyz.shape)

# Save mask as NIfTI (uint8 labels)
mask_img = nib.Nifti1Image(mask_xyz.astype(np.uint8), ref_affine, header=ref_header)
mask_img.set_data_dtype(np.uint8)

out_path = OUT_PATH
nib.save(mask_img, out_path)
print("Saved:", out_path)

ref shape: (512, 512, 671) mask shape: (512, 512, 671)
Saved: /Users/nicolas/Desktop/mask1.nii.gz
